# 00 · Download the open datasets (DDTI + TN3K)

Two of the four datasets are openly downloadable. This notebook fetches them into
`data/raw/extracted/` with the exact folder names the preprocessing scripts expect.

| Dataset | Source | Lands in |
|---|---|---|
| **TN3K** | Google Drive (TRFE-Net bundle, via `gdown`) | `data/raw/extracted/TRFE-Net-for-thyroid-nodule-segmentation-main/picture/` |
| **DDTI** | CIMA@LAB / Kaggle mirror | `data/raw/extracted/DDTI-Pedraza_Digital_Database_Thyroid_SPIE/` |

> The other two datasets (ThyroidXL, Stanford AIMI) are **gated** — see `01_place_gated_datasets.ipynb`.

In [ ]:
# Always run from the repository root so every relative path resolves.
import os
from pathlib import Path
while not (Path.cwd() / 'pyproject.toml').exists() and Path.cwd() != Path.cwd().parent:
    os.chdir('..')
assert (Path.cwd() / 'pyproject.toml').exists(), 'run this notebook from inside the repo'
print('repo root:', Path.cwd())

### TN3K

The cell below downloads and extracts. If you would rather check where it will write
before committing to the transfer, run it once with `--dry-run` appended.


In [ ]:
!python 00_setup/_lib/download_tn3k.py


### DDTI

Same idea. DDTI's direct URL is the less stable of the two; if it has moved, the script
fails with the official page and the mirror to try, and you re-run it with
`--url <zip_url>` or `--zip <local_archive>`.


In [ ]:
!python 00_setup/_lib/download_ddti.py


### Verify
Both extracted folders should now exist.

In [ ]:
import os
for name in ['TRFE-Net-for-thyroid-nodule-segmentation-main',
             'DDTI-Pedraza_Digital_Database_Thyroid_SPIE']:
    p = Path('data/raw/extracted') / name
    print(('OK  ' if p.exists() else 'MISS'), p)

---

## Foundation-model code and weights

The promptable models are third-party. Their code is not vendored here and their weights
are not redistributed, so this is the one place you install both. Only needed if you plan
to run the foundation experiments (2, 3, 5, 6, and the foundation halves of 1 and 4); the
CNN baselines need none of it.

| Model | Code | Weights | Place at |
|---|---|---|---|
| SAM (ViT-H) | [segment-anything](https://github.com/facebookresearch/segment-anything) | `sam_vit_h_4b8939.pth` from the repo's checkpoint links | `pretrained_models/sam/` |
| SAM2 | [sam2](https://github.com/facebookresearch/sam2) | `sam2.1_hiera_large.pt` via the repo's `checkpoints/download_ckpts.sh` | `pretrained_models/sam2/` |
| SAM3 | [sam3](https://github.com/facebookresearch/sam3) | **gated** - request access, then place as `sam3.pt` | `pretrained_models/sam3_vanilla/` |
| MedSAM | uses the `segment_anything` registry | `medsam_vit_b.pth` from the [MedSAM](https://github.com/bowang-lab/MedSAM) checkpoint drive | `pretrained_models/medsam/` |
| MedSAM-2 | [MedSAM2](https://github.com/bowang-lab/MedSAM2) (a `sam2` fork) | `MedSAM2_latest.pt`, also on [HF](https://huggingface.co/wanglab/MedSAM2) | `pretrained_models/medsam2/` |

SAM3 is the only gated one: access must be requested from its authors, exactly as for the
two gated datasets. Everything else is a public download.

`pretrained_models/` is git-ignored. Each wrapper in `thyroidbench/models/` names the exact
file it loads and raises a clear `FileNotFoundError` naming the missing path, so you can
also just run an experiment and let it tell you what it wants.


In [ ]:
# Install the three packages that are not on PyPI, pinned to the revisions used for the
# reported results. Uncomment to run; each is a few hundred MB of build + download.

# !pip install "git+https://github.com/facebookresearch/segment-anything.git"
# !pip install "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"

# MedSAM-2 is a sam2 fork and must land at this exact path: medsam2_wrapper.py puts it on
# sys.path ahead of the upstream sam2 package, which is what fixes the feature-size
# mismatch at 512 px.
# !git clone https://github.com/bowang-lab/MedSAM2.git pretrained_models/medsam2_fork
# !git -C pretrained_models/medsam2_fork checkout 332f30d
# !pip install -e pretrained_models/medsam2_fork


### Check what is in place

Reports which model packages import and which checkpoints are present. Missing entries are
only a problem for the experiments that use that model.


In [ ]:
import importlib.util
from pathlib import Path

CKPTS = {
    'SAM (ViT-H)': ('segment_anything', 'pretrained_models/sam/sam_vit_h_4b8939.pth'),
    'SAM2':        ('sam2',             'pretrained_models/sam2/sam2.1_hiera_large.pt'),
    'SAM3':        ('sam3',             'pretrained_models/sam3_vanilla/sam3.pt'),
    'MedSAM':      ('segment_anything', 'pretrained_models/medsam/medsam_vit_b.pth'),
    'MedSAM-2':    ('sam2',             'pretrained_models/medsam2/MedSAM2_latest.pt'),
}
print(f"{'model':<13}{'package':<10}{'weights':<10}path")
print('-' * 78)
for name, (pkg, ckpt) in CKPTS.items():
    has_pkg = importlib.util.find_spec(pkg) is not None
    has_ckpt = Path(ckpt).exists()
    print(f"{name:<13}{'yes' if has_pkg else 'no':<10}{'yes' if has_ckpt else 'no':<10}{ckpt}")
